In [144]:
from pymatgen.core.structure import Structure
from pymatgen.io.vasp import Poscar
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.transformations.standard_transformations import RotationTransformation
from pymatgen.io.ase         import AseAtomsAdaptor
from ase.io.vasp             import write_vasp
import numpy as np
import os
import json

In [111]:
# Define name of folder and path to reference POSCAR
general_folder = 'input/CeO2-heterostructure'

# Step 1: Read the POSCAR files and best terminations for each one
substrate_miller = (3, 1, 1)
substrate_POSCAR = "/Users/cibran/work/UPC/SlabOptimization/input/CeO2/slab_v0/bulk/POSCAR"
substrate_structure = Structure.from_file(substrate_POSCAR)  # Load the first surface slab

film_miller = (4, 4, 3)
film_POSCAR = "/Users/cibran/work/UPC/SlabOptimization/input/CeO2/slab_v0/bulk/POSCAR"
film_structure = Structure.from_file(film_POSCAR)  # Load the second surface slab

In [112]:
if not os.path.exists(general_folder):
    os.system(f'mkdir {general_folder}')

heterostructure_data = {
        'substrate_miller': substrate_miller,
        'film_miller': film_miller,
        'gap': 2, # Gap between film and substrate
        'vacuum_over_film': 20, # Vacuum over the top of the film
        'film_thickness': 1, # Film thickness
        'substrate_thickness': 1, # Substrate thickness
        'in_layers': True # Set the thickness in layer units
    }

with open(f'{general_folder}/heterostructure_data.json', 'w') as json_file:
    json.dump(heterostructure_data, json_file)

# Copy POSCARs there
os.system(f'cp {substrate_POSCAR} {general_folder}/POSCAR-substrate')
os.system(f'cp {film_POSCAR}      {general_folder}/POSCAR-film')

0

In [148]:
# Step 2: Rotate slab_2 around the c-axis (z-axis)
twist_angle = 15  # Rotation angle in degrees
rotation = RotationTransformation(axis=[0, 0, 1], angle=twist_angle)  # Rotate around z-axis
film_structure = rotation.apply_transformation(film_structure)

In [149]:
# Step 2: Initialize CoherentInterfaceBuilder
interface_builder = CoherentInterfaceBuilder(
    substrate_structure=substrate_structure,  # First slab (substrate)
    film_structure=film_structure,       # Second slab (film)
    substrate_miller=substrate_miller,  # Miller index of the substrate surface
    film_miller=film_miller       # Miller index of the film surface
)

In [150]:
# Step 3: Generate possible interfaces
for t_idx, termination in enumerate(interface_builder._terminations.keys()):
    termination_folder = f'{general_folder}/termination-{t_idx}'
    if not os.path.exists(termination_folder):
        os.system(f'mkdir {termination_folder}')

    # Generate heterostructures with given termination
    interfaces = interface_builder.get_interfaces(
        termination=termination,  # Or 'bottom', should compare them
        gap=2, # Gap between film and substrate
        vacuum_over_film=20, # Vacuum over the top of the film
        film_thickness=1, # Film thickness
        substrate_thickness=1, # Substrate thickness
        in_layers=True # Set the thickness in layer units
    )

    for idx, interface in enumerate(list(interfaces)):
        idx_folder = f'{termination_folder}/{idx}'
        if not os.path.exists(idx_folder):
            os.system(f'mkdir {idx_folder}')

        # Save slab structure into miller_folder
        write_vasp(f'{idx_folder}/POSCAR', AseAtomsAdaptor.get_atoms(interface), direct=True, sort=True)